# Домашнее задание 3 (13 pts)

## Задача 1 (4 pts). Обучение: конечные разности vs reverse AD

**Напоминание про SGD.** *Stochastic Gradient Descent* — итерационный метод минимизации функции потерь $L(\mathbf{w})$. На шаге $k$ берём оценку градиента $g_k$ (по всему датасету — полный градиент, чаще по случайному минибатчу — стохастическую) и обновляем параметры:

$$
\mathbf{w}_{k\ +\ 1} = \mathbf{w}_k - \alpha_k\, g_k,
$$

где $\alpha_k>0$ — размер шага (learning rate). Качество и стоимость метода напрямую зависят от того, **как** считают $g_k$: наивно через конечные разности или через автоматическое дифференцирование.

На практике градиент для SGD почти всегда берут reverse AD. Ниже предлагается сравнить его с наивной альтернативой — **центральными конечными разностями** — на задаче бинарной классификации.

Данные: синтетический датасет (`sklearn.datasets.make_classification` или свой генератор), $m\ge 1000$ объектов, $d\in\{50, 100\}$.
Модель: логистическая регрессия

$$
L(\mathbf{w})
=
\frac1m\sum_{i\ =\ 1}^m
\log\bigl(1+\exp(-y_i\mathbf{w}^\top\mathbf{x}_i)\bigr)
+\frac{\mu}{2}\|\mathbf{w}\|_2^2,
\qquad y_i\in\{\pm 1\},\;\mu=10^{-2}.
$$

**Задание**

1. **(1 pts)** Эффективно реализуй `grad_fd(w)` центральными разностями по каждой координате (FD) и `grad_ad(w)` через autograd (AD).

2. **(1 pts)** В одной точке $\mathbf{w}_0$ сравни:
   - относительную ошибку $\|\nabla_{\mathrm{fd}}-\nabla_{\mathrm{ad}}\|_2/\|\nabla_{\mathrm{ad}}\|_2$ для нескольких $\varepsilon$ (например, $10^{-2},\ldots,10^{-8}$);
   - время одного полного градиента FD vs AD.

   Построй график ошибки от $\varepsilon$ (ожидаем U-образный график: слишком большое $\varepsilon$ — ошибка аппроксимации, слишком малое — шум от одинарной точности представления числа).

3. **(2 pts)** Запусти **один и тот же** SGD (одинаковый $\mathbf{w}_0$, шаг, число итераций $K\sim 100\text{–}300$) с градиентом, посчитанным с помощью AD и FD (возьми наиболее подходящий $\varepsilon$ из прошлого пункта).
   - На одном графике покажи зависимость $L(\mathbf{w}_k)$ от $k$ и от времени работы.
   - Прокомментируй, какой подход быстрее, во сколько раз и почему. Сравни теоретические оценки с наблюдениями. Также оцени насколько сильно деградирует сходимость при использовании FD-подхода.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

# from sklearn.datasets import make_classification
# import torch

# TODO: данные, loss, grad_fd, grad_ad, SGD, графики

## Задача 2 (4 pts). Когда нужен forward-подход

Reverse-подход выгоден, когда **выходов мало, а дифференцируемых входов много** (типичный случай: скалярный лосс и много параметров сети).
Forward/JVP выгоден, когда **выходов много, а дифференцируемых входов мало**.

Рассмотрим обычный MLP $\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x})\in\mathbb{R}^{m}$:
$\mathbb{R}^{n}\xrightarrow{\text{Linear}}\mathbb{R}^{h}\xrightarrow{\tanh}\mathbb{R}^{h}\xrightarrow{\text{Linear}}\mathbb{R}^{m}$,
где $\boldsymbol{\theta}\in\mathbb{R}^{P}$ — **все** веса и bias модели, вход $\mathbf{x}$ зафиксирован.
Нужно оценить полный якобиан отображения $\boldsymbol{\theta}\mapsto\mathbf{f}_{\boldsymbol{\theta}}(\mathbf{x})$:

$$
J=\frac{\partial\mathbf{f}}{\partial\boldsymbol{\theta}}\in\mathbb{R}^{m\ \times\ P}.
$$

Стоимость: forward-сборка $\sim P$ JVP, reverse-сборка $\sim m$ VJP. Режим выбирают по соотношению $P$ и $m$ (а не по $n$ и $m$ сами по себе).

**Задание**

1. **(1 pts)** Реализуй два способа собрать $J$ по параметрам модели:
   - **reverse:** `jacrev` (или $m$ VJP по координатам выхода);
   - **forward:** `jacfwd` (или $P$ JVP с базисом по параметрам).

   Удобно работать с **плоским** вектором параметров длины $P$ (см. заготовку ниже).

2. **(2 pts)** Зафиксируй размерность входного вектора и варьируй число параметров модели и размерность выходного вектора. Рассмотри три пары, в которых $m > P$, и три пары, в которых $m < P$.

   Для каждой пары измерь время forward- и reverse-сборки $J$ ($\ge 10$ прогонов после warmup). Таблица или heatmap: $\mathrm{time_{rev}}/\mathrm{time_{fwd}}$ и отдельно отношение $P/m$.

3. **(1 pts)** Сформулируй правило выбора `jacfwd` vs `jacrev` для якобиана по параметрам и подтверди его числами ($P$ vs $m$). Если для данной MLP reverse почти всегда быстрее, объясни почему (как $P$ растёт с $m$) и при необходимости добавь свои конфигурации.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.func import functional_call, jacfwd, jacrev

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
HIDDEN = 64


class MLP(nn.Module):
    """Обычный MLP: R^n -> R^m, один скрытый слой ширины `hidden`.
    """

    def __init__(self, n_in: int, n_out: int, hidden: int = HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden),
            nn.Tanh(),
            nn.Linear(hidden, n_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (n_in,)
        return self.net(x)


def make_model_and_input(n: int, m: int, seed: int = 0):
    torch.manual_seed(seed)
    model = MLP(n_in=n, n_out=m, hidden=HIDDEN).to(DEVICE)
    model.eval()
    x = torch.randn(n, device=DEVICE, dtype=torch.float32)
    return model, x


def num_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def pack_params(model: nn.Module) -> torch.Tensor:
    """Плоский вектор параметров"""
    return torch.cat([p.reshape(-1) for p in model.parameters()])


def unpack_params(model: nn.Module, flat: torch.Tensor) -> dict:
    params = {}
    offset = 0
    for name, p in model.named_parameters():
        n = p.numel()
        params[name] = flat[offset : offset + n].reshape(p.shape)
        offset += n
    return params


def f_of_params(model: MLP, flat: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    return functional_call(model, unpack_params(model, flat), (x,))


def jac_forward(model: MLP, x: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError


def jac_reverse(model: MLP, x: torch.Tensor) -> torch.Tensor:
    raise NotImplementedError


def time_jacobian(jac_fn, model, x, n_warmup=3, n_runs=10):
    """Среднее время одного вызова jac_fn(model, x)."""
    def _sync():
        if DEVICE == "cuda":
            torch.cuda.synchronize()

    for _ in range(n_warmup):
        _ = jac_fn(model, x)
        _sync()

    ts = []
    for _ in range(n_runs):
        _sync()
        t0 = time.perf_counter()
        J = jac_fn(model, x)
        _sync()
        ts.append(time.perf_counter() - t0)
        del J
    return float(np.mean(ts)), float(np.std(ts))


In [ ]:
# Место для вашего решения

## Задача 3 (5 pts). Чек-пойнтинг на практике

Reverse-подход хранит активации (результаты вычисления промежуточных функций), что приводит к чрезмерному потреблению памяти при обучении глубоких моделей.

Подход **Gradient checkpointing** (rematerialization): сохраняем только часть активаций, остальные пересчитываем при вычислении градиентов. При этом потребление памяти падает, но растёт время расчётов, так как необходимо пересчитывать промежуточные значения повторно.

Возьми **глубокую** MLP-модель (или небольшую свёрточную сеть) с $L\ge 20$ блоками вида Linear $\to$ activation (GELU/ReLU), ширина $w\sim 512\text{–}1024$, batch $B\sim 32\text{–}128$, вход разумной размерности (посмотри на размеры объектов в датасетах CIFAR10/ImageNet).
Целевая функция — [cross-entropy](https://docs.pytorch.org/docs/2.14/generated/torch.nn.CrossEntropyLoss.html) на синтетике/CIFAR-подвыборке (достаточно одного forward + backward).

Инструменты:
- PyTorch: `torch.utils.checkpoint.checkpoint` / `checkpoint_sequential`;
- JAX: `jax.checkpoint` (`remat`).

**Задание**

1. **(1 pts)** Реализуй две версии модели, которые вычисляют одно и то же:
   - `no_ckpt` — обычный forward;
   - `ckpt` — чек-пойнтинг сегментами (например, каждые $k$ слоёв или `checkpoint_sequential(..., segments=S)`).

2. **(2 pts)** Для нескольких глубин $L\in\{16,32,48,64\}$ (или пока влезает в память) измерь на **одном** шаге `loss.backward()` (или JAX-аналоге):
   - пиковую память активаций/аллокаций
     (`torch.cuda.max_memory_allocated` на GPU или `tracemalloc` / `torch.mps` / оценку через `torch.profiler` на CPU);
   - время forward + backward.

   Нарисуй два графика зависимости $L$ от памяти и времени для `no_ckpt` и `ckpt` версий модели. Особенно интересны случаи, когда без чек-пойнтинга возникает ошибка ООМ, а с чек-пойнтингом удаётся выполнить forward + backward.

3. **(1 pts)** Поварьируй число сегментов / частоту чек-пойнтов при фиксированной $L$ (например, $S\in\{2,4,8,16\}$). Покажи trade-off «память — время» (нарисуй две кривые).

4. **(1 pts)** Прокомментируй полученные результаты: для каких значений (глубина, batch, device) нужно по умолчанию включать чек-пойнтинг и какой оверхед по времени можно ожидать?

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

import torch